# SFT eval on Colab — one-shot run

Runs `training/evaluate.py` on `policy_0` (raw base) vs `policy_1` (SFT adapter in `ckpts/sft/`) on 20 val layouts × 4 samples.

**How to use**: `Runtime → Run all`. Walk away for ~20 min. Come back to the table at the end.


In [ ]:
!pip install -q 'transformers>=4.45' 'trl>=0.11' 'peft>=0.13' 'accelerate>=0.34' 'bitsandbytes>=0.43' 'datasets>=2.20' 'pydantic>=2.7'
!git clone https://github.com/Andrii238/ml-project.git /content/ml_project
%cd /content/ml_project


In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
!python -m training.evaluate \
  --checkpoints policy_0=BASE policy_1=./ckpts/sft \
  --samples-per-layout 4 --n-val 20 \
  --out results/eval_sft_vs_base.json


In [ ]:
import json, pandas as pd
d = json.load(open('results/eval_sft_vs_base.json'))
rows = [{'ckpt': c['name'],
         'composite': round(c['mean_composite'], 4),
         'green_sci/s': round(c['mean_green_science'], 4),
         'valid %': round(c['valid_output_pct'], 1),
         'parse_ok %': round(c['parse_ok_pct'], 1),
         'materials': round(c['mean_materials'], 1),
         'cells': round(c['mean_cells'], 1),
         'machines': round(c['mean_machines'], 2)} for c in d]
pd.DataFrame(rows).set_index('ckpt')
